# 📊 Spotify Streaming History – Gesamtübersicht

Überblick über alle Streaming-Daten: Gesamtminuten, Anzahl Tracks/Artists, Jahresvergleich.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from config import *
from data_loader import load_data, get_music, get_podcasts, ensure_results_dir

plt.style.use(PLOT_STYLE)
plt.rcParams['figure.dpi'] = PLOT_DPI
plt.rcParams['savefig.dpi'] = PLOT_DPI
plt.rcParams['savefig.bbox'] = 'tight'

In [ ]:
df = load_data()
music = get_music(df)
podcasts = get_podcasts(df)

## Gesamtstatistiken

In [ ]:
total_hours = df['hours_played'].sum()
total_days = total_hours / 24
total_tracks = music['track'].nunique()
total_artists = music['artist'].nunique()
total_albums = music['album'].nunique()
total_streams = len(music)
year_range = f"{df['year'].min()} – {df['year'].max()}"

print(f"{'='*50}")
print(f"  Spotify Streaming History: {year_range}")
print(f"{'='*50}")
print(f"  🎵 Gesamte Hörzeit:    {total_hours:,.0f} Stunden ({total_days:,.1f} Tage)")
print(f"  ▶️  Streams (Musik):    {total_streams:,}")
print(f"  🎤 Unique Artists:      {total_artists:,}")
print(f"  🎶 Unique Tracks:       {total_tracks:,}")
print(f"  💿 Unique Alben:        {total_albums:,}")
print(f"  🎙️ Podcast-Streams:     {len(podcasts):,}")
print(f"{'='*50}")

## Hörzeit pro Jahr (Stunden)

In [ ]:
yearly = music.groupby('year').agg(
    hours=('hours_played', 'sum'),
    streams=('track', 'count'),
    unique_artists=('artist', 'nunique'),
    unique_tracks=('track', 'nunique'),
).reset_index()

fig, ax = plt.subplots(figsize=PLOT_FIGSIZE_WIDE)
bars = ax.bar(yearly['year'].astype(str), yearly['hours'], color=COLOR_PRIMARY, edgecolor='white', linewidth=0.5)
for bar, h in zip(bars, yearly['hours']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, f'{h:,.0f}h',
            ha='center', va='bottom', fontweight='bold', fontsize=10)
ax.set_xlabel('Jahr')
ax.set_ylabel('Stunden')
ax.set_title('🎵 Gehörte Stunden pro Jahr', fontsize=16, fontweight='bold')
ax.set_ylim(0, yearly['hours'].max() * 1.15)

out = ensure_results_dir('overview')
fig.savefig(out / 'hours_per_year.png')
plt.show()
print(f"Gespeichert: {out / 'hours_per_year.png'}")

## Streams & Unique Artists pro Jahr

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=PLOT_FIGSIZE_WIDE)

# Streams pro Jahr
axes[0].bar(yearly['year'].astype(str), yearly['streams'], color=COLOR_PALETTE[0])
axes[0].set_title('Streams pro Jahr', fontweight='bold')
axes[0].set_ylabel('Anzahl Streams')
for i, v in enumerate(yearly['streams']):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontsize=9)

# Unique Artists pro Jahr
axes[1].bar(yearly['year'].astype(str), yearly['unique_artists'], color=COLOR_PALETTE[10])
axes[1].set_title('Unique Artists pro Jahr', fontweight='bold')
axes[1].set_ylabel('Anzahl Artists')
for i, v in enumerate(yearly['unique_artists']):
    axes[1].text(i, v + 5, f'{v:,}', ha='center', fontsize=9)

plt.suptitle('📈 Jahresvergleich', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(out / 'streams_artists_per_year.png')
plt.show()

## Jahresübersicht als Tabelle

In [ ]:
yearly_display = yearly.copy()
yearly_display.columns = ['Jahr', 'Stunden', 'Streams', 'Unique Artists', 'Unique Tracks']
yearly_display['Stunden'] = yearly_display['Stunden'].map('{:,.1f}'.format)
yearly_display['Streams'] = yearly_display['Streams'].map('{:,}'.format)
yearly_display['Unique Artists'] = yearly_display['Unique Artists'].map('{:,}'.format)
yearly_display['Unique Tracks'] = yearly_display['Unique Tracks'].map('{:,}'.format)
yearly_display

## Musik vs. Podcasts Anteil

In [ ]:
type_hours = pd.DataFrame({
    'Typ': ['Musik', 'Podcasts'],
    'Stunden': [music['hours_played'].sum(), podcasts['hours_played'].sum()]
})

fig, ax = plt.subplots(figsize=(8, 8))
colors = [COLOR_PRIMARY, COLOR_PALETTE[10]]
wedges, texts, autotexts = ax.pie(
    type_hours['Stunden'], labels=type_hours['Typ'], autopct='%1.1f%%',
    colors=colors, startangle=90, textprops={'fontsize': 14}
)
for t in autotexts:
    t.set_fontweight('bold')
ax.set_title('🎵 Musik vs. 🎙️ Podcasts', fontsize=16, fontweight='bold')
fig.savefig(out / 'music_vs_podcasts.png')
plt.show()